# V5W_06 - Chiusura funzionale: Alpha sensorimotoria x Phenotype Index

La firma forte della tesi (EEG_40) era l'**alpha fronto-centrale/mu** (AUC 0.91). Qui: sui 5words, l'**alpha sensorimotoria per-soggetto** correla con il **Phenotype Index** (posizione sull'asse di connettivita C0->C1, il continuum di V5W_05)?

**Non-circolare:** l'alpha (spettro) non entra mai nel clustering (connettivita). Se correlano -> connettivita e spettro sono la stessa dimensione, replicata su coorte indipendente -> chiude il Cap.6.

**Env: `daniele_311`**. Richiede le cache di V5W_05 (`feat_abs_pcc.npz`, `cluster_labels.npz`).

## par.1 - Config + regione mu

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w06')
project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
V5W05    = project_root / 'models' / 'v5w05'
CKPT_DIR = project_root / 'models' / 'v5w06'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CSV_ROOT = project_root / 'data' / '5words_subjects'
N_CHAN, FS = 61, 256
triu_idx = np.triu_indices(N_CHAN, k=1)

import mne
ELOC = project_root / 'src' / 'io' / 'ebneuro.locs'
_RENAME = {'T3':'T7','T4':'T8','T5':'P7','T6':'P8'}; _BAD = {'A1','A2'}
_mont = mne.channels.read_custom_montage(str(ELOC), coord_frame='head')
CHAN_NAMES = [_RENAME.get(c, c) for c in _mont.ch_names if c not in _BAD][:N_CHAN]
SENSORIMOTOR = ['C5','C3','C1','Cz','C2','C4','C6','CP5','CP3','CP1','CPz','CP2','CP4','CP6']
REGION_IDX = [CHAN_NAMES.index(c) for c in SENSORIMOTOR if c in CHAN_NAMES]
log.info(f'Regione mu: {len(REGION_IDX)} canali')
assert V5W05.exists(), 'models/v5w05 mancante — esegui prima V5W_05 (par. 2-4)'


## par.2 - Phenotype Index dalle cache V5W_05

In [ ]:
# Phenotype Index (PI) per soggetto — dalle cache di V5W_05 (asse interno C1-C0)
_feat = np.load(V5W05 / 'feat_abs_pcc.npz', allow_pickle=True)
FEAT_G, SUBJ = _feat['feat_g'], _feat['subj'].tolist()
_lab = np.load(V5W05 / 'cluster_labels.npz', allow_pickle=True)
LAB_SUBJ, LAB = _lab['subj'].tolist(), _lab['labels']

def vec_to_sym(v):
    M = np.zeros((N_CHAN, N_CHAN)); M[triu_idx] = v; return M + M.T
NS = np.array([vec_to_sym(FEAT_G[i]).sum(1) / (N_CHAN - 1) for i in range(len(SUBJ))])  # (S,61)
lab_map = {s: int(l) for s, l in zip(LAB_SUBJ, LAB)}
lab_arr = np.array([lab_map[s] for s in SUBJ])
diff = NS[lab_arr == 1].mean(0) - NS[lab_arr == 0].mean(0)
diff = diff / (np.linalg.norm(diff) + 1e-12)
_pi = NS @ diff
PI = {s: float(p) for s, p in zip(SUBJ, _pi)}
log.info(f'PI per {len(PI)} soggetti  range=[{min(PI.values()):+.3f}, {max(PI.values()):+.3f}]')


## par.3 - Alpha sensorimotoria per soggetto (CSV img)

In [ ]:
# Alpha relativa sensorimotoria per soggetto (dai CSV img) — non-circolare
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
subj_csv = defaultdict(list)
for sd in sorted(CSV_ROOT.iterdir()):
    m = _PAT.match(sd.name)
    if not m: continue
    for csv in sd.glob('*_img_*.csv'):
        if not csv.name.startswith('._'): subj_csv[int(m.group(1))].append(csv)

CACHE = CKPT_DIR / 'subject_alpha.npz'
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True); ALPHA = {int(s): float(a) for s, a in zip(z['subj'], z['alpha'])}
    log.info(f'Alpha da cache: {len(ALPHA)} soggetti')
else:
    def subj_alpha(csvs):
        psd_sum, n = None, 0
        for p in csvs:
            x = pd.read_csv(p, header=None).values.astype(np.float32)
            f, Pxx = welch(x, fs=FS, nperseg=256, axis=1)
            psd_sum = Pxx if psd_sum is None else psd_sum + Pxx; n += 1
        psd = psd_sum / n
        bp = lambda lo, hi: psd[:, (f >= lo) & (f < hi)].sum(1)
        rel = bp(8, 13) / (bp(1, 45) + 1e-12)
        return float(rel[REGION_IDX].mean())
    ALPHA = {}
    for sid in tqdm(sorted(subj_csv), desc='alpha/soggetto'):
        ALPHA[sid] = subj_alpha(subj_csv[sid])
    np.savez(CACHE, subj=np.array(list(ALPHA)), alpha=np.array(list(ALPHA.values())))
    log.info(f'Alpha calcolata, cache salvata: {len(ALPHA)} soggetti')


## par.4 - Correlazione (chiusura funzionale)

In [ ]:
# Correlazione PI (connettivita) vs alpha sensorimotoria (spettro)
common = sorted(set(PI) & set(ALPHA))
x = np.array([PI[s] for s in common])
y = np.array([ALPHA[s] for s in common])
r, p_r = pearsonr(x, y); rho, p_s = spearmanr(x, y)

fig, ax = plt.subplots(figsize=(7.5, 6))
sc = ax.scatter(x, y, c=x, cmap='RdBu_r', s=90, edgecolor='k', linewidth=0.5, zorder=3)
b1, b0 = np.polyfit(x, y, 1); xs = np.linspace(x.min(), x.max(), 50)
ax.plot(xs, b1 * xs + b0, 'k--', lw=1.5, alpha=0.7)
for s, xi, yi in zip(common, x, y):
    if xi < np.percentile(x, 8) or xi > np.percentile(x, 92):
        ax.annotate(f'P{s}', (xi, yi), fontsize=7, alpha=0.7)
ax.set_xlabel('Phenotype Index (asse connettivita C0->C1)')
ax.set_ylabel('Alpha relativa sensorimotoria (mu, C/CP)')
ax.set_title(f'V5W_06 - Connettivita x Spettro\nPearson r={r:.3f} (p={p_r:.1e})   Spearman rho={rho:.3f} (p={p_s:.1e})',
             fontweight='bold')
plt.colorbar(sc, ax=ax, label='PI'); plt.tight_layout()
plt.savefig(FIG_DIR / 'v5w06_alpha_vs_PI.png', dpi=160, bbox_inches='tight'); plt.show()

verdict = 'CHIUSURA FUNZIONALE: connettivita e alpha mu = stessa dimensione (non-circolare, replica EEG_40)' if p_r < 0.05 else 'Nessuna correlazione: su questa coorte alpha e asse di connettivita NON coincidono'
print('='*60)
print(f'  V5W_06 - Alpha mu x Phenotype Index ({len(common)} soggetti)')
print('='*60)
print(f'  Pearson  r = {r:+.3f}  (p = {p_r:.2e})')
print(f'  Spearman rho = {rho:+.3f}  (p = {p_s:.2e})')
print(f'  -> {verdict}')
print('='*60)
